# Synapse serverless — prueba de integración (centraal-dataframework 0.2.0)

Valida contra un ambiente real lo que agrega `SynapseClient` / `ctx.synapse`:

| # | Prueba | Requiere |
|---|---|---|
| 1 | Driver ODBC y `pyodbc` | — |
| 2 | Conexión con identidad (`SELECT 1`) | `SYNAPSE_WORKSPACE` o `SYNAPSE_SERVER` |
| 3 | Identidad, base y motor (serverless = `EngineEdition` 11) | — |
| 4 | Parámetros `?` en serverless (supuesto a confirmar) | — |
| 5 | Catálogo: bases y objetos visibles | — |
| 6 | Lectura de una tabla/vista (`scalar`, `query`, `read_df`) | `SYNAPSE_TEST_TABLE` |
| 7 | Cambio de base (`for_database`) | `SYNAPSE_TEST_OTHER_DB` |
| 8 | Plantillas `{{param}}` con lista blanca | — |
| 9 | DDL con `execute_file` y lotes `GO` (crea y borra una vista) | `SYNAPSE_TEST_DDL_SCHEMA` |
| 10 | Error de SQL no se reintenta | — |
| 11 | Uso desde una tarea (`TaskContext.synapse`, sin storage) | — |

Primero siga la sección **Instrucciones para ejecutar** (abajo). Las pruebas sin su variable opcional
quedan como `SALTADO`. Al final hay un resumen; esa celda falla si alguna prueba falló.

## Instrucciones para ejecutar

Siga los pasos en orden. Los comandos están para **PowerShell (Windows)**; las diferencias para Linux/macOS van indicadas.

### Paso 1 — Entorno de Python (3.10 a 3.14)

Desde la raíz del repo `centraal-dataframework`:

```powershell
py -3.12 -m venv .venv-synapse
.venv-synapse\Scripts\Activate.ps1          # Linux/macOS: source .venv-synapse/bin/activate
python -m pip install --upgrade pip
python -m pip install -e ".[synapse]" jupyter ipykernel
```

Con Poetry, la alternativa es `poetry install --extras synapse --with dev` (el grupo `dev` trae Jupyter).

### Paso 2 — Driver ODBC 18 del sistema operativo

`pyodbc` necesita el driver **ODBC Driver 18 for SQL Server** instalado en la máquina (no es un paquete de pip).

- **Windows:** instalador de Microsoft: <https://learn.microsoft.com/sql/connect/odbc/download-odbc-driver-for-sql-server>
- **macOS:** `brew tap microsoft/mssql-release https://github.com/Microsoft/homebrew-mssql-release` y luego `HOMEBREW_ACCEPT_EULA=Y brew install msodbcsql18`
- **Linux:** seguir la guía de la misma página para su distribución (paquete `msodbcsql18`).

Comprobar:

```powershell
python -c "import pyodbc; print(pyodbc.drivers())"   # debe incluir 'ODBC Driver 18 for SQL Server'
```

### Paso 3 — Iniciar sesión en Azure

El notebook se autentica con `DefaultAzureCredential`: en una máquina de desarrollo usa la sesión de Azure CLI. No se necesitan claves.

```powershell
az login                                   # si hay varios tenants: az login --tenant <tenant-id>
az account get-access-token --resource https://database.windows.net/ --query expiresOn
```

Si el segundo comando devuelve una fecha, el token para Synapse funciona.

### Paso 4 — Permitir su IP en el firewall del workspace

```powershell
$ip = Invoke-RestMethod https://api.ipify.org      # Linux/macOS: ip=$(curl -s https://api.ipify.org)
az synapse workspace firewall-rule create --name "dev-$env:USERNAME" --workspace-name <workspace> --resource-group <resource-group> --start-ip-address $ip --end-ip-address $ip
```

Requiere permisos sobre el workspace; si no los tiene, pida a quien lo administra que agregue la IP.

### Paso 5 — Usuario en las bases de Synapse

Un administrador de Synapse ejecuta, **en cada base** que se va a probar (`SYNAPSE_DATABASE` y `SYNAPSE_TEST_OTHER_DB`):

```sql
CREATE USER [usuario@dominio.com] FROM EXTERNAL PROVIDER;
ALTER ROLE db_datareader ADD MEMBER [usuario@dominio.com];
-- Si las tablas o vistas leen el lago con una credencial de base de datos:
-- GRANT REFERENCES ON DATABASE SCOPED CREDENTIAL::[<credencial>] TO [usuario@dominio.com];
-- Solo para la prueba 9 (DDL), en el esquema de SYNAPSE_TEST_DDL_SCHEMA:
-- GRANT CREATE VIEW TO [usuario@dominio.com];
-- GRANT ALTER ON SCHEMA::[<esquema>] TO [usuario@dominio.com];
```

En la Function App se usa el nombre de la identidad administrada en lugar del correo.

### Paso 6 — Variables del ambiente (`.env.synapse`)

```powershell
Copy-Item docs\synapse.env.example .env.synapse     # Linux/macOS: cp docs/synapse.env.example .env.synapse
```

Edite `.env.synapse` (está en `.gitignore`, no se versiona):

| Variable | Obligatoria | Ejemplo | Para qué |
|---|---|---|---|
| `SYNAPSE_WORKSPACE` | sí (o `SYNAPSE_SERVER`) | `synw-mi-ambiente` | Usa el endpoint serverless `<workspace>-ondemand.sql.azuresynapse.net` |
| `SYNAPSE_DATABASE` | recomendada | `master` o una base de lectura | Base por defecto de las pruebas 2–6 |
| `SYNAPSE_AUTH` | no | `identity` | Dejar `identity` (usa `az login`) |
| `SYNAPSE_TEST_TABLE` | no | `esquema.tabla` | Prueba 6: lectura real de datos |
| `SYNAPSE_TEST_OTHER_DB` | no | `silver` | Prueba 7: cambio de base |
| `SYNAPSE_TEST_DDL_SCHEMA` | no | `pruebas` | Prueba 9: crea y borra una vista temporal. Vacía = solo lectura |

Las pruebas con su variable vacía quedan como `SALTADO`, no como falla.

### Paso 7 — Ejecutar

- **VS Code / Jupyter:** abrir este notebook, elegir como kernel el Python de `.venv-synapse` y ejecutar **Run All**.
- **Sin interfaz:**

  ```powershell
  jupyter nbconvert --to notebook --execute docs/synapse_integracion.ipynb --output synapse_integracion_resultado.ipynb
  ```

- **Solo las pruebas básicas con pytest** (usa el mismo `.env.synapse`): `pytest -m synapse -v`

### Paso 8 — Leer el resultado

La celda **Resumen** muestra una tabla con `OK` / `SALTADO` / `FALLA` por prueba y **falla si alguna prueba falló**. Si algo falla, vea la sección *Solución de problemas* al final.

> **Importante:** las salidas de las pruebas 5 y 6 muestran nombres de objetos y filas del ambiente. **No versione el notebook con salidas**: limpie las salidas antes de commitear (*Clear All Outputs*). El archivo `*_resultado.ipynb` de `nbconvert` está en `.gitignore`.

## 0. Configuración del ambiente

In [ ]:
import logging
import os
import sys
import time
import uuid
from pathlib import Path

from dotenv import load_dotenv

# Raíz del repo (el notebook vive en docs/)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT))

ENV_FILE = ROOT / '.env.synapse'
if ENV_FILE.exists():
    load_dotenv(ENV_FILE, override=True)
    print(f'Variables cargadas de {ENV_FILE}')
else:
    print(f'No existe {ENV_FILE}; se usan solo las variables del entorno.')

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

VARIABLES = [
    'SYNAPSE_WORKSPACE', 'SYNAPSE_SERVER', 'SYNAPSE_DATABASE', 'SYNAPSE_AUTH', 'SYNAPSE_USER',
    'SYNAPSE_QUERY_TIMEOUT', 'SYNAPSE_TEST_TABLE', 'SYNAPSE_TEST_OTHER_DB', 'SYNAPSE_TEST_DDL_SCHEMA',
]
for name in VARIABLES:
    print(f'{name:26} = {os.environ.get(name) or "-"}')
print(f'{"SYNAPSE_PASSWORD":26} = {"(definida)" if os.environ.get("SYNAPSE_PASSWORD") else "-"}')

if not (os.environ.get('SYNAPSE_WORKSPACE') or os.environ.get('SYNAPSE_SERVER')):
    raise RuntimeError('Defina SYNAPSE_WORKSPACE (serverless) o SYNAPSE_SERVER en .env.synapse (ver Paso 6 de las instrucciones)')

TEST_TABLE = os.environ.get('SYNAPSE_TEST_TABLE', '').strip()
OTHER_DB = os.environ.get('SYNAPSE_TEST_OTHER_DB', '').strip()
DDL_SCHEMA = os.environ.get('SYNAPSE_TEST_DDL_SCHEMA', '').strip()

In [ ]:
resultados = {}


def check(nombre, fn, requiere=True):
    """Ejecuta una prueba y registra OK / FALLA / SALTADO."""
    if not requiere:
        resultados[nombre] = 'SALTADO'
        print(f'[SALTADO] {nombre}')
        return None
    inicio = time.monotonic()
    try:
        valor = fn()
    except Exception as exc:  # noqa: BLE001
        resultados[nombre] = f'FALLA: {type(exc).__name__}: {exc}'
        print(f'[FALLA] {nombre}: {type(exc).__name__}: {exc}')
        return None
    resultados[nombre] = 'OK'
    print(f'[OK] {nombre} ({time.monotonic() - inicio:.1f}s)')
    return valor

## 1. Driver ODBC y pyodbc

In [ ]:
def driver_disponible():
    import pyodbc

    drivers = pyodbc.drivers()
    print('pyodbc', pyodbc.version, '| drivers:', drivers)
    assert 'ODBC Driver 18 for SQL Server' in drivers, 'Instale ODBC Driver 18 for SQL Server'


check('1. driver ODBC 18 + pyodbc', driver_disponible)

## 2. Cliente y conexión

In [ ]:
from centraal_dataframework import SynapseClient
from centraal_dataframework.settings import clear_settings_cache, get_settings
from centraal_dataframework.synapse import get_synapse, render

clear_settings_cache()
client = get_synapse(get_settings())
print(client)  # password y credencial no se muestran
print('serverless:', client.server.endswith('-ondemand.sql.azuresynapse.net'))


def select_1():
    assert client.scalar('SELECT 1') == 1


check('2. conexión SELECT 1', select_1)

## 3. Identidad, base y motor

In [ ]:
def identidad():
    fila = client.query(
        "SELECT SUSER_SNAME() AS usuario, DB_NAME() AS base, "
        "CAST(SERVERPROPERTY('EngineEdition') AS int) AS engine_edition"
    )[0]
    print(fila)
    if client.server.endswith('-ondemand.sql.azuresynapse.net'):
        assert fila['engine_edition'] == 11, 'Se esperaba Synapse serverless (EngineEdition 11)'
    return fila


identidad_actual = check('3. identidad y motor', identidad)

## 4. Parámetros `?` en serverless

Supuesto del documento de arquitectura: serverless acepta parámetros en `SELECT`. Si falla, use
`render` con `{{param}}` también para lecturas.

In [ ]:
def parametros():
    assert client.query('SELECT ? AS texto, ? AS numero', 'a', 7) == [{'texto': 'a', 'numero': 7}]


check('4. parámetros ?', parametros)

## 5. Catálogo visible para la identidad

In [ ]:
def catalogo():
    bases = [r['name'] for r in client.query('SELECT name FROM sys.databases ORDER BY name')]
    print('bases:', bases)
    objetos = client.read_df(
        "SELECT TOP 30 s.name AS esquema, o.name AS objeto, o.type_desc "
        "FROM sys.objects o JOIN sys.schemas s ON s.schema_id = o.schema_id "
        "WHERE o.type IN ('V', 'U') ORDER BY s.name, o.name"
    )
    display(objetos)  # noqa: F821
    return bases


bases = check('5. catálogo', catalogo)

## 6. Lectura de la tabla de prueba (`SYNAPSE_TEST_TABLE`)

In [ ]:
def lectura():
    n = client.scalar(render('SELECT COUNT_BIG(*) FROM {{tabla}}', tabla=TEST_TABLE))
    print('filas:', n)
    muestra = client.read_df(render('SELECT TOP 10 * FROM {{tabla}}', tabla=TEST_TABLE))
    display(muestra)  # noqa: F821
    filas = client.query(render('SELECT TOP 3 * FROM {{tabla}}', tabla=TEST_TABLE))
    assert isinstance(filas, list) and (not filas or isinstance(filas[0], dict))
    assert len(muestra) == min(n, 10)


check('6. lectura tabla de prueba', lectura, requiere=bool(TEST_TABLE))

## 7. Otra base con `for_database` (`SYNAPSE_TEST_OTHER_DB`)

In [ ]:
def otra_base():
    original = client.database
    otro = client.for_database(OTHER_DB)
    assert otro.scalar('SELECT DB_NAME()') == OTHER_DB
    assert client.database == original  # el cliente original no cambia
    print('original:', client.database, '| nuevo:', otro.database)


check('7. for_database', otra_base, requiere=bool(OTHER_DB))

## 8. Plantillas `{{param}}` con lista blanca (sin red)

In [ ]:
def plantillas():
    sql = render("SELECT * FROM {{esquema}}.t WHERE fecha = '{{lote}}'", esquema='demo', lote='2026-10-09')
    print(sql)
    for malo in ["x'; DROP VIEW v; --", 'a b', 'a--b']:
        try:
            render('SELECT {{v}}', v=malo)
        except ValueError as exc:
            print('rechazado:', exc)
        else:
            raise AssertionError(f'debía rechazar {malo!r}')


check('8. render con lista blanca', plantillas)

## 9. DDL con `execute_file` y lotes `GO` (`SYNAPSE_TEST_DDL_SCHEMA`)

Crea una vista temporal `<esquema>.cdf_prueba_<id>`, la lee y la borra. Requiere permiso de
`CREATE VIEW` / `ALTER` en ese esquema. No escribe datos en el lago.

In [ ]:
import tempfile


def ddl():
    run = uuid.uuid4().hex[:8]
    sql_file = Path(tempfile.mkdtemp()) / 'vista_prueba.sql'
    sql_file.write_text(
        "CREATE OR ALTER VIEW {{esquema}}.cdf_prueba_{{run}} AS SELECT 1 AS x, '{{run}}' AS run\n"
        "GO\n"
        "CREATE OR ALTER VIEW {{esquema}}.cdf_prueba_{{run}} AS SELECT 2 AS x, '{{run}}' AS run\n",
        encoding='utf-8',
    )
    vista = f'{DDL_SCHEMA}.cdf_prueba_{run}'
    try:
        client.execute_file(sql_file, esquema=DDL_SCHEMA, run=run)
        assert client.query(f'SELECT x, run FROM {vista}') == [{'x': 2, 'run': run}]
        print('vista creada y leída:', vista)
    finally:
        client.execute(f'DROP VIEW IF EXISTS {vista}')
        print('vista borrada')


check('9. execute_file + GO', ddl, requiere=bool(DDL_SCHEMA))

## 10. Un error de SQL falla de inmediato (sin reintentos)

In [ ]:
def error_sql():
    import pyodbc

    inicio = time.monotonic()
    try:
        client.query('SELECT * FROM esquema_que_no_existe.tabla_que_no_existe')
    except pyodbc.Error as exc:
        duracion = time.monotonic() - inicio
        print(f'{type(exc).__name__} en {duracion:.1f}s: {exc}')
        assert duracion < 30, 'no debía reintentar con espera'
    else:
        raise AssertionError('se esperaba un error de SQL')


check('10. error de SQL sin reintento', error_sql)

## 11. Uso desde una tarea del framework

`ctx.synapse` se crea al primer uso; una tarea que solo usa Synapse no necesita storage configurado.

In [ ]:
from centraal_dataframework import TaskContext


def desde_tarea():
    def sin_storage():
        raise AssertionError('la tarea no debía crear el datalake')

    ctx = TaskContext(
        name='prueba_synapse',
        run_id=uuid.uuid4().hex,
        logger=logging.getLogger('prueba_synapse'),
        params={'lote': '2026-10-09'},
        datalake_factory=sin_storage,
    )
    assert ctx.synapse.scalar('SELECT 1') == 1
    assert ctx.synapse is ctx.synapse
    if OTHER_DB:
        assert ctx.synapse_for(OTHER_DB).scalar('SELECT DB_NAME()') == OTHER_DB


check('11. TaskContext.synapse', desde_tarea)

## Resumen

In [ ]:
import pandas as pd

resumen = pd.DataFrame([{'prueba': k, 'resultado': v} for k, v in resultados.items()])
display(resumen)  # noqa: F821
fallas = [k for k, v in resultados.items() if v.startswith('FALLA')]
print(f"OK={sum(v == 'OK' for v in resultados.values())} "
      f"SALTADO={sum(v == 'SALTADO' for v in resultados.values())} FALLA={len(fallas)}")
assert not fallas, f'Pruebas fallidas: {fallas}'

## Solución de problemas

| Síntoma (mensaje) | Causa probable | Qué hacer |
|---|---|---|
| `SynapseNoConfigurado: Falta pyodbc` | No se instaló el extra | Paso 1: `pip install -e ".[synapse]"` |
| `IM002` / `Data source name not found` / prueba 1 falla | Falta el driver ODBC 18 | Paso 2 |
| `DefaultAzureCredential failed to retrieve a token` | No hay sesión de Azure CLI | Paso 3: `az login` |
| `Client with IP address '...' is not allowed` (40615) | IP fuera del firewall | Paso 4 |
| `Login failed for user '<token-identified principal>'` (18456) | El usuario no existe en esa base o el tenant es otro | Paso 5; `az login --tenant <tenant-id>`. No reintente en ráfaga: la cuenta se puede bloquear |
| `Cannot open database "..." requested by the login` | La base no existe o no hay usuario en ella | Revisar `SYNAPSE_DATABASE` / `SYNAPSE_TEST_OTHER_DB` y el Paso 5 |
| Prueba 3 falla con `EngineEdition` distinto de 11 | No está conectado al endpoint serverless | Usar `SYNAPSE_WORKSPACE` sin `SYNAPSE_SERVER` |
| Prueba 4 falla | Serverless no acepta parámetros `?` en ese caso | Anotarlo: usar `render` con `{{param}}` también en lecturas |
| Prueba 6 falla con `Invalid object name` | Nombre de tabla mal escrito o en otra base | Usar `esquema.tabla` de la base `SYNAPSE_DATABASE` (o `base.esquema.tabla`) |
| Prueba 6 falla con `Cannot find the CREDENTIAL` o `... is not accessible` | Sin permiso sobre la credencial o el lago | Paso 5: `GRANT REFERENCES ON DATABASE SCOPED CREDENTIAL` |
| Prueba 9 falla con `permission denied` | Sin `CREATE VIEW` / `ALTER` en el esquema | Paso 5 o dejar `SYNAPSE_TEST_DDL_SCHEMA` vacía |
| Una consulta tarda y termina en timeout | Consulta pesada; límite de serverless 30 min | Usar una tabla de prueba pequeña o ajustar `SYNAPSE_QUERY_TIMEOUT` |